# TFP-BD Vehicle-Type Classification: ResNet50 vs MobileNetV3

Methodology-r 8 ta model er moddhe 2 ta classifier. Same 70/20/10 split (Sharmin er Data Preparation):

- Prottek annotation box theke chobi kete (crop, 10% padding, 224x224) 9-class classification
- Train: class proti shorboccho 8,000 crop (Car/Rickshaw/Bus onek beshi, tai cap), validation: class proti 3,000
- **Test: SOB test crop** (kono cap nai) - honest result
- Best epoch validation macro-F1 diye bachai, test e ekbar
- Result: Accuracy, macro Precision/Recall/F1, class-wise F1, confusion matrix, speed

**Input:** shudhu `Data Preparation` (Sharmin). **Settings:** GPU T4 x2, **Internet ON** (ImageNet pretrained weight download er jonno).
Cell 1 chalie check koro, tarpor **Save Version -> Save & Run All**. Somoy ~40-60 min.

In [ ]:
# CELL 1: setup + dataset khuje ber kora
import os, time, random, json, gc, zipfile, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision
import torchvision.transforms as T
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, f1_score
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEV, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(GPU nai!)")
print("torch", torch.__version__, "| torchvision", torchvision.__version__)

CLASSES = ["Car", "Bus", "Truck", "Motorcycle", "Bicycle", "Rickshaw", "Mini-Truck", "CNG", "Pedestrian"]
WK, OUTD = "/kaggle/working", "/kaggle/working/cls_results"
os.makedirs(OUTD, exist_ok=True)
EXTS = (".jpg", ".jpeg", ".png", ".bmp")

TRAIN_CAP, VAL_CAP = 8000, 3000     # class proti crop (train / validation); test e kono cap nai
MIN_SIDE_PX = 6                      # er cheye choto box baad (pixel)
IMG = 224
BATCH = 128
EPOCHS = {"ResNet50": 6, "MobileNetV3-Large": 8}
LR = {"ResNet50": 3e-4, "MobileNetV3-Large": 1e-3}
NUM_WORKERS = 4

SH = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "tfp_dataset" in dirs:
        c = os.path.join(root, "tfp_dataset")
        if all(os.path.isdir(os.path.join(c, s, "images")) for s in ("train", "val", "test")):
            SH = c
            break
    dirs[:] = [d for d in dirs if d not in ("images", "labels", "Bangladeshi Traffic Flow Dataset")]
assert SH, "tfp_dataset pawa jayni - Add Input e Sharmin er 'Data Preparation' notebook jog koro"
print("Dataset:", SH)

# Internet check: pretrained weight namano jay kina
try:
    torchvision.models.mobilenet_v3_large(weights="IMAGENET1K_V2")
    print("Pretrained weight download OK (Internet ON)")
except Exception as e:
    raise RuntimeError("Pretrained weight namano gelo na - Settings e Internet ON koro. " + str(e)[:200])

In [ ]:
# CELL 2: annotation box theke crop er list (index) banano + class proti cap
def index_split(split):
    rows, tiny = [], 0
    img_dir, lbl_dir = f"{SH}/{split}/images", f"{SH}/{split}/labels"
    for f in sorted(os.listdir(img_dir)):
        if not f.lower().endswith(EXTS):
            continue
        stem = os.path.splitext(f)[0]
        lp = os.path.join(lbl_dir, stem + ".txt")
        if not os.path.exists(lp):
            continue
        with open(lp) as fh:
            for line in fh:
                q = line.split()
                if len(q) < 5:
                    continue
                c, xc, yc, w, h = int(q[0]), *map(float, q[1:5])
                # 640x480 dhore choto box baad (ashol size load er somoy hisab hoy)
                if w * 640 < MIN_SIDE_PX or h * 480 < MIN_SIDE_PX:
                    tiny += 1
                    continue
                rows.append((os.path.join(img_dir, f), c, xc, yc, w, h))
    return rows, tiny

def cap_per_class(rows, cap, seed=SEED):
    rng = random.Random(seed)
    by = {}
    for r in rows:
        by.setdefault(r[1], []).append(r)
    out = []
    for c in sorted(by):
        lst = by[c][:]
        rng.shuffle(lst)
        out += lst[:cap] if cap else lst
    rng.shuffle(out)
    return out

ALL = {}
for s in ("train", "val", "test"):
    ALL[s], tiny = index_split(s)
    print(f"{s}: {len(ALL[s])} crop ({tiny} ta {MIN_SIDE_PX}px er choto box baad)")

DATA = {"train": cap_per_class(ALL["train"], TRAIN_CAP),
        "val": cap_per_class(ALL["val"], VAL_CAP),
        "test": ALL["test"]}

cnt = pd.DataFrame({f"{s}_all": pd.Series([r[1] for r in ALL[s]]).value_counts() for s in ALL})
for s in DATA:
    cnt[f"{s}_used"] = pd.Series([r[1] for r in DATA[s]]).value_counts()
cnt = cnt.reindex(range(9)).fillna(0).astype(int)
cnt.index = CLASSES
cnt.to_csv(f"{OUTD}/crop_counts.csv")
print("\nCrop count (all = mot, used = cap er pore):")
print(cnt.to_string())

In [ ]:
# CELL 3: Dataset + DataLoader (crop on-the-fly, disk e kichu likhi na)
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

class CropDS(Dataset):
    def __init__(self, rows, train):
        self.rows, self.train = rows, train
        aug = [T.RandomHorizontalFlip(), T.ColorJitter(0.2, 0.2, 0.2)] if train else []
        self.tf = T.Compose(aug + [T.ToTensor(), T.Normalize(MEAN, STD)])
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        p, c, xc, yc, w, h = self.rows[i]
        img = Image.open(p).convert("RGB")
        W, H = img.size
        pad = random.uniform(0.0, 0.2) if self.train else 0.1
        bw, bh = w * W, h * H
        x0 = max(0, (xc * W - bw / 2) - pad * bw); y0 = max(0, (yc * H - bh / 2) - pad * bh)
        x1 = min(W, (xc * W + bw / 2) + pad * bw); y1 = min(H, (yc * H + bh / 2) + pad * bh)
        if x1 - x0 < 2 or y1 - y0 < 2:
            x0, y0, x1, y1 = 0, 0, W, H
        crop = img.crop((int(x0), int(y0), int(x1), int(y1))).resize((IMG, IMG), Image.BILINEAR)
        return self.tf(crop), c

def loader(split, shuffle):
    return DataLoader(CropDS(DATA[split], train=(split == "train")), batch_size=BATCH, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True, drop_last=(split == "train"))

LOADERS = {"train": loader("train", True), "val": loader("val", False), "test": loader("test", False)}
xb, yb = next(iter(LOADERS["val"]))
print("Batch shape:", tuple(xb.shape), "| labels:", yb[:10].tolist())

# class weight (train used count er sqrt inverse) - rare class e beshi guruttho
n_used = np.array([max(1, (np.array([r[1] for r in DATA["train"]]) == c).sum()) for c in range(9)], dtype=np.float32)
CLASS_W = torch.tensor((n_used.max() / n_used) ** 0.5, dtype=torch.float32)
CLASS_W = CLASS_W / CLASS_W.mean()
print("Class weight:", dict(zip(CLASSES, CLASS_W.numpy().round(2))))

In [ ]:
# CELL 4: model, train, evaluate function
def build_model(name):
    if name == "ResNet50":
        m = torchvision.models.resnet50(weights="IMAGENET1K_V2")
        m.fc = nn.Linear(m.fc.in_features, 9)
    elif name == "MobileNetV3-Large":
        m = torchvision.models.mobilenet_v3_large(weights="IMAGENET1K_V2")
        m.classifier[3] = nn.Linear(m.classifier[3].in_features, 9)
    else:
        raise ValueError(name)
    return m.to(DEV)

try:
    make_scaler = lambda: torch.amp.GradScaler("cuda")
    make_scaler()
except Exception:
    make_scaler = lambda: torch.cuda.amp.GradScaler()

@torch.no_grad()
def predict(model, split):
    model.eval()
    ys, ps = [], []
    for x, y in LOADERS[split]:
        with torch.autocast("cuda", dtype=torch.float16, enabled=DEV.type == "cuda"):
            out = model(x.to(DEV, non_blocking=True))
        ps.append(out.argmax(1).cpu().numpy())
        ys.append(y.numpy())
    return np.concatenate(ys), np.concatenate(ps)

def train_model(name):
    model = build_model(name)
    epochs, steps = EPOCHS[name], len(LOADERS["train"])
    opt = torch.optim.AdamW(model.parameters(), lr=LR[name], weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR[name], total_steps=epochs * steps, pct_start=0.15)
    crit = nn.CrossEntropyLoss(weight=CLASS_W.to(DEV), label_smoothing=0.1)
    scaler = make_scaler()
    best_f1, hist, t0 = -1, [], time.time()
    ckpt = f"{WK}/{name}_best.pt"
    for ep in range(1, epochs + 1):
        model.train()
        run_loss, n = 0.0, 0
        for i, (x, y) in enumerate(LOADERS["train"]):
            x, y = x.to(DEV, non_blocking=True), y.to(DEV, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.float16, enabled=DEV.type == "cuda"):
                loss = crit(model(x), y)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            run_loss += loss.item() * len(y)
            n += len(y)
        yv, pv = predict(model, "val")
        vf1 = f1_score(yv, pv, average="macro")
        vacc = accuracy_score(yv, pv)
        hist.append({"epoch": ep, "train_loss": run_loss / n, "val_acc": vacc, "val_macro_f1": vf1,
                     "minutes": (time.time() - t0) / 60})
        tag = ""
        if vf1 > best_f1:
            best_f1 = vf1
            torch.save(model.state_dict(), ckpt)
            tag = "  <- best"
        print(f"[{name}] epoch {ep}/{epochs} | loss {run_loss / n:.4f} | val acc {vacc * 100:.2f}% | "
              f"val macro-F1 {vf1 * 100:.2f}% | {(time.time() - t0) / 60:.1f} min{tag}")
    pd.DataFrame(hist).to_csv(f"{OUTD}/history_{name}.csv", index=False)
    model.load_state_dict(torch.load(ckpt, map_location=DEV))
    return model, hist, (time.time() - t0) / 60
print("Functions ready")

In [ ]:
# CELL 5: ResNet50 train (~15-20 min)
MODELS = {}
MODELS["ResNet50"] = train_model("ResNet50")

In [ ]:
# CELL 6: MobileNetV3-Large train (~8-12 min)
MODELS["MobileNetV3-Large"] = train_model("MobileNetV3-Large")

In [ ]:
# CELL 7: TEST set e ekbar evaluation + speed + confusion matrix
@torch.no_grad()
def speed_ms(model, n=200):
    model.eval()
    xs = [CropDS(DATA["test"][i:i + 1], train=False)[0][0].unsqueeze(0).to(DEV) for i in range(n)]
    for x in xs[:20]:
        model(x)
    if DEV.type == "cuda":
        torch.cuda.synchronize()
    t = time.time()
    for x in xs:
        model(x)
    if DEV.type == "cuda":
        torch.cuda.synchronize()
    return (time.time() - t) / n * 1000

rows, per_class = [], []
fig, axes = plt.subplots(1, 2, figsize=(18, 7.5))
for ax, (name, (model, hist, train_min)) in zip(axes, MODELS.items()):
    yt, pt = predict(model, "test")
    p, r, f, s = precision_recall_fscore_support(yt, pt, labels=list(range(9)), zero_division=0)
    best = max(hist, key=lambda h: h["val_macro_f1"])
    rows.append({"model": name, "params_M": round(sum(q.numel() for q in model.parameters()) / 1e6, 2),
                 "ms_per_crop_bs1": round(speed_ms(model), 2), "train_minutes": round(train_min, 1),
                 "best_epoch": best["epoch"], "val_macro_f1_%": best["val_macro_f1"] * 100,
                 "test_n": len(yt), "test_accuracy_%": accuracy_score(yt, pt) * 100,
                 "test_macro_precision_%": p.mean() * 100, "test_macro_recall_%": r.mean() * 100,
                 "test_macro_f1_%": f.mean() * 100,
                 "test_weighted_f1_%": f1_score(yt, pt, average="weighted") * 100})
    for c in range(9):
        per_class.append({"model": name, "class": CLASSES[c], "support": int(s[c]),
                          "precision_%": p[c] * 100, "recall_%": r[c] * 100, "f1_%": f[c] * 100})
    cm = confusion_matrix(yt, pt, labels=list(range(9)))
    pd.DataFrame(cm, index=CLASSES, columns=CLASSES).to_csv(f"{OUTD}/confusion_{name}.csv")
    cmn = cm / cm.sum(1, keepdims=True).clip(min=1)
    ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(9)); ax.set_xticklabels(CLASSES, rotation=45, ha="right")
    ax.set_yticks(range(9)); ax.set_yticklabels(CLASSES)
    for i in range(9):
        for j in range(9):
            ax.text(j, i, f"{cmn[i, j] * 100:.0f}", ha="center", va="center", fontsize=9,
                    color="white" if cmn[i, j] > 0.6 else "black")
    ax.set_xlabel("Predicted"); ax.set_ylabel("Annotation")
    ax.set_title(f"{name} (test, row %)")
plt.tight_layout()
plt.savefig(f"{OUTD}/confusion_matrices.png", dpi=110, bbox_inches="tight")
plt.show()

res = pd.DataFrame(rows)
pc = pd.DataFrame(per_class)
res.to_csv(f"{OUTD}/classifier_results.csv", index=False)
pc.to_csv(f"{OUTD}/classifier_per_class.csv", index=False)
print("=" * 80 + "\nCLASSIFICATION (test set, sob crop)\n" + "=" * 80)
print(res.round(2).T.to_string())
print("\nClass-wise F1 (%):")
print(pc.pivot(index="class", columns="model", values="f1_%").reindex(CLASSES).round(1).to_string())

In [ ]:
# CELL 8: sob result ek zip e
zp = f"{WK}/cls_results.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(os.listdir(OUTD)):
        z.write(os.path.join(OUTD, f), f)
print("Saved:", zp, f"({os.path.getsize(zp) / 1e6:.2f} MB)")